# Advanced RAG Lab — Week 2 Thursday

**Wednesday taught why naive RAG fails. Today we fix it.**

We build a RAG pipeline step by step, adding one upgrade at a time:

| Step | What we build | Why |
|---|---|---|
| 1 | Naive RAG | Baseline to compare against |
| 2 | RAG with citations | User knows where the answer came from |
| 3 | Conversational RAG | Handles follow-up questions |
| 4 | RAGAS evaluation | Measures if any of it actually works |

**We skip LangSmith tracing** in this notebook — it requires an API key signup and environment setup that wastes class time. The concept is covered in the notes at the end.

---
**Stack we use today:**
- `langchain` + `langchain-community` — RAG pipeline
- `langchain-huggingface` — HuggingFace embeddings (official dedicated package)
- `langchain-text-splitters` — text chunking utilities
- `groq` + `langchain-groq` — free LLM (same as Week 1)
- `faiss-cpu` — local vector store, no signup needed
- `sentence-transformers` — free local embeddings
- `ragas` — evaluation framework
- `rank-bm25` — sparse keyword search for hybrid

---
## Part 1 — Install Everything

**Why these specific versions?**
- `langchain==0.3.*` — stable release series; LCEL is mature, all integrations are stable
- `langchain-huggingface` — the **official** package for HuggingFace embeddings; `langchain_community.embeddings.HuggingFaceEmbeddings` is deprecated
- `langchain-text-splitters` — text splitters were extracted into their own package; the old `langchain.text_splitter` import is deprecated
- `ragas>=0.3,<0.4` — v0.3 has a stable `evaluate()` API with `SingleTurnSample`/`EvaluationDataset`; v0.4 is a major experimental rewrite that breaks this API

> **Restart runtime after this cell finishes** if running in Colab for the first time.

In [ ]:
# Install all required packages with compatible versions
# Run this cell once; restart the runtime afterwards in Colab

!pip install -q \
  "langchain>=0.3,<0.4" \
  "langchain-core>=0.3,<0.4" \
  "langchain-community>=0.3,<0.4" \
  langchain-groq \
  langchain-huggingface \
  langchain-text-splitters \
  faiss-cpu \
  rank-bm25 \
  sentence-transformers \
  groq \
  "ragas>=0.3,<0.4"

print("Done. All libraries installed.")

---
## Part 2 — Imports and API Key

Paste your Groq API key below.
Get one free at: https://console.groq.com/keys

**Import notes (what changed from older notebooks):**
- `RecursiveCharacterTextSplitter` → now from `langchain_text_splitters` (dedicated package)
- `Document` → now from `langchain_core.documents` (not `langchain.schema`)
- `HuggingFaceEmbeddings` → now from `langchain_huggingface` (not `langchain_community.embeddings`)
- Everything else (`FAISS`, `BM25Retriever`, `EnsembleRetriever`, `ChatGroq`, LCEL) is unchanged

In [ ]:
import os
import json
from groq import Groq

# ── Text splitting ──────────────────────────────────────────────────────────
# FIX: was `from langchain.text_splitter import ...` — that module is deprecated.
# The text splitters were extracted into their own package: langchain-text-splitters.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Core document schema ────────────────────────────────────────────────────
# FIX: was `from langchain.schema import Document` — that path is deprecated.
# Document now lives in langchain_core.documents.
from langchain_core.documents import Document

# ── LangChain core (unchanged) ──────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Vector store and retrieval ──────────────────────────────────────────────
from langchain_community.vectorstores import FAISS          # unchanged
from langchain_community.retrievers import BM25Retriever     # unchanged
from langchain.retrievers import EnsembleRetriever           # unchanged

# ── Embeddings ──────────────────────────────────────────────────────────────
# FIX: was `from langchain_community.embeddings import HuggingFaceEmbeddings`.
# LangChain moved HuggingFace integrations into their own dedicated package.
from langchain_huggingface import HuggingFaceEmbeddings

# ── LLM ────────────────────────────────────────────────────────────────────
from langchain_groq import ChatGroq  # unchanged

# ── Set your Groq API key ───────────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = "paste_your_groq_key_here"

print("Imports done.")

---
## Part 3 — Our Knowledge Base

We need documents for the RAG system to search through.

In a real product this would be PDFs, company docs, a database.
For class we use a fake company HR policy — short enough to reason about,
but with enough variety to show where naive RAG breaks.

**Notice**: Some questions will need multi-hop reasoning (two chunks).
Some will have keyword mismatches. These are the failure modes from Wednesday.

---
### 🔍 What is Naive RAG?
**Naive RAG** is the simplest form: embed a query → retrieve top-k chunks → stuff them into a prompt → ask an LLM to answer.
It works surprisingly well for simple factual questions, but fails when:
- The query uses different words than the document (semantic mismatch)
- The answer requires combining two chunks (multi-hop)
- The user asks a follow-up question (no memory)
- You need to know *which document* the answer came from (no citations)

In [ ]:
# Fake company HR policy documents
# Each string becomes one document in our knowledge base

raw_documents = [
    """
    Leave Policy
    Employees are entitled to 20 days of annual leave per year.
    Sick leave is capped at 10 days per year with a medical certificate required after 3 consecutive days.
    Maternity leave is 90 days fully paid. Paternity leave is 14 days.
    Unused annual leave can be carried forward to the next year up to a maximum of 10 days.
    """,
    """
    Reimbursement and Expense Policy
    All reimbursement requests must be submitted within 30 days of the expense.
    Travel expenses require pre-approval from the line manager.
    Meals during business travel are covered up to Rs. 2,000 per day.
    Equipment purchases above Rs. 10,000 require CFO approval.
    """,
    """
    Remote Work Policy
    Employees may work remotely up to 3 days per week with manager approval.
    Remote work is not permitted during the first 3 months of employment.
    A stable internet connection and a dedicated workspace are required.
    Core hours for remote employees are 10am to 4pm PKT.
    """,
    """
    Team Structure
    Hamza Khan manages the Data Engineering team.
    Sara Ahmed manages the Product team.
    The Engineering department head is Bilal Rana.
    All team leads report directly to the department head.
    """,
    """
    Salary and Compensation
    Junior engineers have a salary band of Rs. 80,000 to Rs. 120,000.
    Senior engineers have a salary band of Rs. 150,000 to Rs. 220,000.
    Team leads and managers have a salary band of Rs. 200,000 to Rs. 300,000.
    Annual increments are processed in July each year.
    """,
    """
    Disciplinary Policy
    Penalties for misconduct follow a three-strike system.
    First offense: written warning.
    Second offense: final warning and performance improvement plan.
    Third offense: termination.
    Contract breach penalties are handled under the employment contract terms.
    """
]

# Convert to LangChain Document objects
# Each Document has page_content (the text) and metadata (source info)
docs = [
    Document(
        page_content=text.strip(),
        metadata={"source": f"HR_Policy_Section_{i+1}"}
    )
    for i, text in enumerate(raw_documents)
]

print(f"Created {len(docs)} documents.")
print(f"\nFirst document preview:")
print(docs[0].page_content[:200])

---
## Part 4 — Chunk the Documents

Wednesday's rule: **content type determines chunking strategy.**

Our documents are prose, so we use the **recursive splitter**.

### Why RecursiveCharacterTextSplitter?
It splits on a priority list of separators: `\n\n` → `\n` → ` ` → `""`.
This means it tries to keep paragraphs together first, then sentences, then words.
For prose documents (like HR policies, articles, documentation), this preserves the
natural meaning boundaries better than a fixed-size character split.

- `chunk_size=300`: each chunk is at most 300 characters
- `chunk_overlap=50`: 50 characters of overlap between chunks

**Why overlap?** If a sentence spans a chunk boundary, overlap ensures
neither chunk loses the context needed to understand it.

In [ ]:
# RecursiveCharacterTextSplitter is the go-to splitter for prose.
# It splits on paragraph → sentence → word boundaries in order of preference,
# preserving semantic coherence better than a naive fixed-character split.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"Documents: {len(docs)} → Chunks after splitting: {len(chunks)}")
print(f"\nExample chunk:")
print(chunks[0].page_content)
print(f"\nChunk source: {chunks[0].metadata['source']}")

---
## Part 5 — Build the Vector Store (Dense Retrieval)

We embed each chunk and store it in FAISS — a local vector database.

### What is Dense Retrieval?
**Dense retrieval** converts both documents and queries into high-dimensional vectors (embeddings),
then finds the nearest neighbors by cosine similarity.
A query like "days off" will find the Leave Policy even though those exact words aren't there —
because "days off" and "annual leave" are semantically close in the vector space.

### Why FAISS?
FAISS (Facebook AI Similarity Search) is an extremely fast vector index that runs **entirely in memory**
with no server, no signup, and no cost. Perfect for a bootcamp lab.
In production you'd use Pinecone, Weaviate, or pgvector.

We use `all-MiniLM-L6-v2` — a small, fast, free embedding model
that runs locally without any API key.

In [ ]:
# Load the embedding model (downloads ~90MB the first time)
print("Loading embedding model... (first run downloads ~90MB)")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build FAISS vector store from our chunks
# This embeds every chunk and stores the vectors in memory
vector_store = FAISS.from_documents(chunks, embeddings)

# Create a retriever from the vector store
# k=4 means: return the 4 most similar chunks
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Vector store built.")

# Quick test
test_results = dense_retriever.invoke("How many days can I take off?")
print(f"\nTest query: 'How many days can I take off?'")
print(f"Top result:")
print(test_results[0].page_content)

---
## Part 6 — Add Sparse Retrieval (BM25)

Wednesday's lesson: dense retrieval fails on **exact keywords**.

### What is Sparse Retrieval?
**Sparse retrieval** uses traditional term-frequency statistics instead of vectors.
It only activates on words that *actually appear* in the query and document.
This makes it excellent for:
- Proper nouns (names, product codes, version numbers)
- Rare technical terms with no semantic neighbours
- Queries like "GPT-4.1" or "Hamza Khan" where meaning is irrelevant — you need the exact token

**BM25 = Best Match 25** — a classical search algorithm, the same one
Elasticsearch uses under the hood.

In [ ]:
# BM25 retriever — pure keyword matching, no embeddings needed
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# Quick test with an exact-keyword query
bm25_results = bm25_retriever.invoke("Hamza Khan")
print("BM25 test query: 'Hamza Khan'")
print(f"Top result: {bm25_results[0].page_content}")

# Compare: dense often misses exact names
dense_results = dense_retriever.invoke("Hamza Khan")
print(f"\nDense test query: 'Hamza Khan'")
print(f"Top result: {dense_results[0].page_content}")

---
## Part 7 — Hybrid Search: Combine Both

### What is Hybrid Search?
Hybrid search merges dense and sparse retrieval scores into a single ranked list.

The formula from Wednesday:

```
hybrid = α × dense_score + (1 − α) × sparse_score
```

LangChain's `EnsembleRetriever` implements this using **Reciprocal Rank Fusion (RRF)**,
which is even more robust than a simple weighted average.
We give each retriever a weight — default is 0.5 each (equal blend).

**Why this almost always beats pure semantic search:**
Real queries mix meaning AND specific keywords.
Dense catches the first. Sparse catches the second.
Hybrid catches both.

In [ ]:
# EnsembleRetriever = hybrid search
# weights=[0.5, 0.5] means equal blend of dense and sparse
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

# Test with a query that has both meaning AND specific keywords
hybrid_results = hybrid_retriever.invoke("What is the salary range for Hamza's team?")

print("Hybrid retrieval test: 'What is the salary range for Hamza\'s team?'")
print(f"\nChunks retrieved: {len(hybrid_results)}")
for i, r in enumerate(hybrid_results):
    print(f"\n--- Chunk {i+1} (source: {r.metadata['source']}) ---")
    print(r.page_content)

---
## Part 8 — Build the LLM

Same Groq + Llama setup as Week 1.
The difference is we now use `ChatGroq` from `langchain_groq`
so it plugs into LangChain chains.

In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# temperature=0 means deterministic output — same input = same output every time.
# For RAG this is what you want: factual, not creative.

# Quick test
response = llm.invoke("Say hello in one sentence.")
print(response.content)

---
## Part 9 — Naive RAG (The Baseline)

Before we build the good version, we build the simple version.
This is what most tutorials show you — and what Wednesday called the starting point.

**How it works:**
1. User asks a question
2. Retrieve relevant chunks
3. Stuff chunks into a prompt
4. Ask the LLM to answer based only on those chunks

**What it's missing:**
- No citations (you can't tell where the answer came from)
- No memory (follow-up questions break it)
- No evaluation (you don't know if it's right)

The chain uses LangChain's **LCEL pipe syntax** (`|`):
read left to right — retrieve → format → prompt → LLM → parse output.

In [ ]:
# Helper function: format retrieved chunks into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Naive RAG prompt
# Note: no citation instruction, no memory, just answer from context
naive_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
""")

# Build the chain using LangChain's pipe syntax
# Read left to right: retrieve → format → prompt → LLM → parse
naive_chain = (
    {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()}
    | naive_prompt
    | llm
    | StrOutputParser()
)

# Test it
answer = naive_chain.invoke("How many days of annual leave do employees get?")
print("Naive RAG answer:")
print(answer)

---
## Part 10 — RAG with Citations

Problem with naive RAG: the user gets an answer but has no idea
which document it came from. They can't verify it.

**Fix:** Return the source alongside the answer.

We do this two ways:
1. Show which chunks were retrieved (always do this)
2. Ask the LLM to reference the source in its answer

In [ ]:
# Citation-aware prompt — instructs the model to mention where it got the info
citation_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.
At the end of your answer, list the sources you used.
If the answer is not in the context, say "I don't know."

Context (each section has a source label):
{context}

Question: {question}

Format your answer as:
Answer: [your answer here]
Sources: [list the source labels you used]
""")


# This version of format_docs includes source labels
def format_docs_with_sources(docs):
    sections = []
    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        sections.append(f"[{source}]\n{doc.page_content}")
    return "\n\n".join(sections)


# Citation chain
citation_chain = (
    {"context": hybrid_retriever | format_docs_with_sources, "question": RunnablePassthrough()}
    | citation_prompt
    | llm
    | StrOutputParser()
)

answer = citation_chain.invoke("What is the salary range for team leads?")
print(answer)

---
## Part 11 — Conversational RAG (Memory)

Wednesday's stateless problem:
```
User: What is the refund policy?
AI:   You must submit within 30 days.
User: How long does it take?
AI:   ??? Take what?
```

The model has no memory. Each call is completely independent.

### What is Conversational RAG / Memory?
**Memory** in a RAG context means injecting past conversation turns into
the prompt so the model can resolve pronouns and references from prior turns.

**The fix is two steps:**
1. **Rewrite** the follow-up question using chat history so it becomes self-contained
2. Use the rewritten question for retrieval

This is the key insight from Wednesday:
**always rewrite before retrieval**, never embed a context-less follow-up.

Why? The retriever doesn't know conversation history — it only sees the query string.
"How long until I can do it?" retrieves nothing useful.
"How long until an employee can work remotely?" retrieves the right chunk.

In [ ]:
# Step 1: Question rewriter
# Given chat history + new question → produce a standalone question

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Given the chat history below and a follow-up question,
rewrite the follow-up question to be a complete standalone question.
Do NOT answer it — only rewrite it.
If it's already standalone, return it unchanged.
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Follow-up question: {question}")
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()


# Test the rewriter with a follow-up question
chat_history = [
    HumanMessage(content="What is the remote work policy?"),
    AIMessage(content="Employees can work remotely up to 3 days per week with manager approval.")
]

rewritten = rewrite_chain.invoke({
    "chat_history": chat_history,
    "question": "How long until I can do it?"
})

print("Original follow-up: 'How long until I can do it?'")
print(f"Rewritten: {rewritten}")

In [ ]:
# Step 2: Full conversational RAG function
# Combines rewriting + retrieval + answer generation

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Answer the question using ONLY the context below.
Include the source label in your answer.
If the answer is not in the context, say "I don't know."

Context:
{context}
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])


def conversational_rag(question, chat_history):
    """
    Full conversational RAG:
    1. Rewrite question with history context
    2. Retrieve chunks using rewritten question
    3. Generate answer with citations
    """

    # Step 1: Rewrite the question
    if chat_history:
        standalone_q = rewrite_chain.invoke({
            "chat_history": chat_history,
            "question": question
        })
    else:
        standalone_q = question

    print(f"[Rewritten query]: {standalone_q}")

    # Step 2: Retrieve using the rewritten question
    retrieved_docs = hybrid_retriever.invoke(standalone_q)
    context = format_docs_with_sources(retrieved_docs)

    # Step 3: Generate answer
    answer_chain = answer_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })

    return answer


# Demo: multi-turn conversation
history = []

# Turn 1
q1 = "What are the rules around remote work?"
print(f"User: {q1}")
a1 = conversational_rag(q1, history)
print(f"AI: {a1}\n")

# Update history
history.append(HumanMessage(content=q1))
history.append(AIMessage(content=a1))

# Turn 2 — ambiguous follow-up
q2 = "What about the salary for that position?"
print(f"User: {q2}")
a2 = conversational_rag(q2, history)
print(f"AI: {a2}")

---
## Part 12 — RAGAS Evaluation

Wednesday's question: **How do you know your RAG is working?**

Manual testing with 5 questions misses hundreds of edge cases in production.
RAGAS measures 4 axes automatically:

| Metric | Question it answers | Target |
|---|---|---|
| Context Precision | Of chunks retrieved, how many were relevant? | ≥ 0.80 |
| Context Recall | Did we get ALL chunks needed to answer? | ≥ 0.80 |
| Answer Relevancy | Does the answer actually address the question? | ≥ 0.85 |
| Faithfulness | Is everything in the answer supported by chunks? | ≥ 0.90 |

**Faithfulness is the most critical.** Low faithfulness = hallucination.

---

### What is RAGAS?
**RAGAS** (Retrieval Augmented Generation Assessment) is an evaluation framework
that uses an LLM-as-a-judge to score your RAG pipeline automatically.
You provide test questions + ground-truth answers, run your pipeline,
then RAGAS scores the quality of both retrieval and generation.

**How RAGAS works (v0.3+ API):**
You give it a list of `SingleTurnSample` objects, each with:
- `user_input`: what was asked
- `response`: what your RAG returned
- `retrieved_contexts`: which chunks were retrieved (list of strings)
- `reference`: the correct answer (you write this)

RAGAS uses an LLM internally to score each row. We configure it to use Groq instead of OpenAI.

> **Note on RAGAS API version:** This notebook uses the RAGAS v0.3 API.
> The key changes from v0.1 (which you may find in older tutorials):
> - Metrics are now **classes** (`Faithfulness()`) not module-level objects (`faithfulness`)
> - Dataset is `EvaluationDataset(samples=[...])` not `Dataset.from_dict({...})`
> - Sample fields are `user_input`, `response`, `retrieved_contexts`, `reference` (not `question`, `answer`, `contexts`, `ground_truth`)

In [ ]:
# RAGAS v0.3+ imports
# FIX: Old notebooks used lowercase module-level instances (faithfulness, answer_relevancy)
# and Dataset.from_dict() from the `datasets` library.
# v0.3 uses class-based metrics and its own EvaluationDataset.

from ragas import evaluate, SingleTurnSample, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Wrap our Groq LLM and local embeddings for RAGAS
# RAGAS needs its own wrapper around LangChain objects
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("RAGAS configured.")

In [ ]:
# Build a small evaluation dataset
# In production you'd have 50-100 test questions.
# For class we use 4 — enough to see the scores.

eval_questions = [
    "How many days of annual leave do employees get?",
    "What are the penalties for misconduct?",
    "What is the salary range for team leads?",
    "How many remote days are allowed per week?",
]

ground_truths = [
    "Employees are entitled to 20 days of annual leave per year.",
    "First offense: written warning. Second: final warning and PIP. Third: termination.",
    "Team leads and managers have a salary band of Rs. 200,000 to Rs. 300,000.",
    "Employees may work remotely up to 3 days per week with manager approval.",
]

# Run our RAG pipeline on each question and collect results
eval_answers = []
eval_contexts = []

for q in eval_questions:
    # Get retrieved chunks
    retrieved = hybrid_retriever.invoke(q)
    contexts = [doc.page_content for doc in retrieved]

    # Get answer
    answer = naive_chain.invoke(q)

    eval_answers.append(answer)
    eval_contexts.append(contexts)

    print(f"Q: {q}")
    print(f"A: {answer[:100]}...\n")

print("Dataset built.")

In [ ]:
# Build the RAGAS dataset using v0.3 API
# FIX: Old code used Dataset.from_dict() from HuggingFace `datasets` library
# and field names: question, answer, contexts, ground_truth.
# v0.3 uses SingleTurnSample + EvaluationDataset with renamed fields:
#   question        → user_input
#   answer          → response
#   contexts        → retrieved_contexts
#   ground_truth    → reference

samples = [
    SingleTurnSample(
        user_input=q,
        response=a,
        retrieved_contexts=c,
        reference=gt
    )
    for q, a, c, gt in zip(eval_questions, eval_answers, eval_contexts, ground_truths)
]

ragas_dataset = EvaluationDataset(samples=samples)

# Run evaluation
# This makes LLM calls for each metric × each row — takes 1-2 minutes
# FIX: Metrics are now instantiated as classes: Faithfulness() not faithfulness
print("Running RAGAS evaluation... (this takes 1-2 minutes)")

results = evaluate(
    dataset=ragas_dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        ContextPrecision(),
        ContextRecall()
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

print("\n=== RAGAS Scores ===")
print(results)

In [ ]:
# Display scores in a readable table with targets

targets = {
    "faithfulness":       0.90,
    "answer_relevancy":   0.85,
    "context_precision":  0.80,
    "context_recall":     0.80
}

print(f"{'Metric':<25} {'Score':>8} {'Target':>8} {'Status':>10}")
print("-" * 55)

scores = results.to_pandas().mean(numeric_only=True)

for metric, target in targets.items():
    if metric in scores:
        score = scores[metric]
        status = "PASS" if score >= target else "NEEDS WORK"
        print(f"{metric:<25} {score:>8.2f} {target:>8.2f} {status:>10}")

print("\nDiagnosis guide:")
print("  Low faithfulness   → add 'only use provided context' to prompt")
print("  Low context recall → chunks too small, or question needs multi-hop")
print("  Low precision      → chunks too large, or K is too high")
print("  Low relevancy      → retriever not finding the right content")

---
## Part 13 — LangSmith Tracing (Concept + Setup)

We're not running this in class — setup takes time and requires an account.
But you need to know what it does and how to wire it in.

**What LangSmith shows you for each query:**
```
query = "What is the refund policy?"
├── embed_query        23 ms
├── vector_search      45 ms   → 4 chunks retrieved
├── build_prompt       (shown in full, so you can debug bad prompts)
├── llm_call         1200 ms   → response
└── total            1318 ms   tokens: 1,450   cost: $0.0043
```

Without it, when your RAG gives a wrong answer you're debugging by guesswork.
With it, you see exactly which chunk was retrieved and exactly what the model was told.

**To set it up (do this after class — takes ~5 minutes):**

```python
# 1. Sign up at smith.langchain.com (free)
# 2. Get your API key
# 3. Add these three lines at the TOP of any LangChain script:

import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_key_here"
os.environ["LANGCHAIN_PROJECT"] = "my-rag-project"

# That's it. Every LangChain call is now automatically traced.
# Open smith.langchain.com to see the dashboard.
```

**Key thing to understand:** You don't change any other code.
Setting those three env variables is the entire integration.
LangChain detects them and sends traces automatically.

LangSmith is **optional** — the rest of this notebook runs fine without it.

---
## Part 14 — What We Built

Here's every component and which Wednesday concept it implements:

| Component | Wednesday concept | Why it matters |
|---|---|---|
| FAISS + HuggingFace embeddings | Dense retrieval | Find by meaning |
| BM25Retriever | Sparse retrieval | Find by exact keywords |
| EnsembleRetriever | Hybrid search | Best of both |
| Question rewriting | Conversational RAG | Memory for follow-ups |
| Source labels in prompt | Citations | Trust and verification |
| RAGAS evaluate() | Evaluation framework | Measure before you ship |
| LangSmith env vars | Observability | Debug without guesswork |

---

## Challenge (Try After Class)

1. Add a 7th document to `raw_documents` — a new policy section
2. Add 2 test questions for it in the RAGAS evaluation dataset
3. Change the chunk size from 300 to 150 — re-run RAGAS and compare scores
4. Change the hybrid weights from `[0.5, 0.5]` to `[0.7, 0.3]` (more dense) — does precision improve?

The scores are your feedback loop. Change one thing, measure, decide.

---

## Quick Reference

```bash
# Chunking defaults
Prose:  RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
Tables: Keep whole — never split
Code:   Split at function boundaries only

# RAGAS targets
faithfulness      >= 0.90   # most critical — catches hallucination
answer_relevancy  >= 0.85
context_precision >= 0.80
context_recall    >= 0.80

# Hybrid search alpha
default: weights=[0.5, 0.5]
keyword-heavy domain: weights=[0.3, 0.7]   (more BM25)
meaning-heavy domain: weights=[0.7, 0.3]   (more dense)
```

**Docs:**
- LangChain: https://python.langchain.com
- RAGAS: https://docs.ragas.io
- LangSmith: https://smith.langchain.com